In [1]:
import torch
import torch.nn as nn

In [ ]:
class MLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_1 = nn.Linear(6,4, bias=False)
    self.layer_2 = nn.Linear(4,2, bias=False)
    self.layer_3 = nn.Linear(2,2, bias=False)

  def forward(self, x):
    x = self.layer_1(x)
    x = self.layer_2(x)
    x = self.layer_3(x)

    return x

In [ ]:
inp = torch.tensor([1,2,3,4,5,6], dtype=torch.float32)
targets = torch.tensor(0)

In [ ]:
torch.manual_seed(123)

mlp = MLP()

In [ ]:
logits = mlp(inp)
print(logits)
loss = nn.functional.cross_entropy(logits, targets)
print(loss)

tensor([0.7440, 0.9682], grad_fn=<SqueezeBackward4>)
tensor(0.8115, grad_fn=<NllLossBackward0>)


In [ ]:
loss.backward()
mlp.layer_1.weight.grad

tensor([[-0.1204, -0.2408, -0.3612, -0.4816, -0.6020, -0.7223],
        [ 0.1231,  0.2463,  0.3694,  0.4925,  0.6157,  0.7388],
        [-0.1258, -0.2516, -0.3774, -0.5032, -0.6290, -0.7548],
        [ 0.0045,  0.0091,  0.0136,  0.0181,  0.0226,  0.0272]])

In [ ]:
mlp.layer_2.weight.grad

tensor([[-0.0117, -0.1271, -0.2434,  0.6620],
        [ 0.0161,  0.1748,  0.3346, -0.9101]])

In [ ]:
mlp.layer_3.weight.grad

tensor([[-0.9769, -0.3865],
        [ 0.9769,  0.3865]])

In [ ]:
mlp.zero_grad()

In [ ]:
#another approach first mat mul the weights of all the layers and then pass the input
mat_mul_output = (mlp.layer_1.weight.T @ mlp.layer_2.weight.T) @ mlp.layer_3.weight.T
mat_mul_output.shape

torch.Size([6, 2])

In [ ]:
logits_2 = inp @ mat_mul_output
logits_2

tensor([0.7440, 0.9682], grad_fn=<SqueezeBackward4>)

In [ ]:
loss_2 = nn.functional.cross_entropy(logits_2, targets)
loss_2

tensor(0.8115, grad_fn=<NllLossBackward0>)

In [ ]:
loss_2.backward()

In [ ]:
mlp.layer_1.weight.grad

tensor([[-0.1204, -0.2408, -0.3612, -0.4816, -0.6020, -0.7223],
        [ 0.1231,  0.2463,  0.3694,  0.4925,  0.6157,  0.7388],
        [-0.1258, -0.2516, -0.3774, -0.5032, -0.6290, -0.7548],
        [ 0.0045,  0.0091,  0.0136,  0.0181,  0.0226,  0.0272]])

In [ ]:
mlp.layer_2.weight.grad

tensor([[-0.0117, -0.1271, -0.2434,  0.6620],
        [ 0.0161,  0.1748,  0.3346, -0.9101]])

In [ ]:
mlp.layer_3.weight.grad

tensor([[-0.9769, -0.3865],
        [ 0.9769,  0.3865]])

- both the approaches yield same result in terms of gradients, logits and loss

In [ ]:
# views, contiguous and transpose
x = torch.tensor([[1,2,3],[4,5,6]])
print(f"x: {x}")
print(f"x shape: {x.shape}")
print(f"x data_ptr: {x.untyped_storage().data_ptr()}")
print(f"is contiguous: {x.is_contiguous()}")

view_x = x.view(3,2)
print(f"view_x : {view_x}")
print(f"view_x shape : {view_x.shape}")
print(f"view_x data_ptr: {view_x.untyped_storage().data_ptr()}")
print(f"is contiguous: {view_x.is_contiguous()}")

y = x.T
print(f"y: {y}")
print(f"y shape: {y.shape}")
print(f"y data_ptr: {y.untyped_storage().data_ptr()}")
print(f"is contiguous: {y.is_contiguous()}")

view_y = y.contiguous().view(2,3)
print(f"view_y : {view_y}")
print(f"view_y shape : {view_y.shape}")
print(f"view_y data_ptr: {view_y.untyped_storage().data_ptr()}")
print(f"is contiguous: {view_y.is_contiguous()}")

x: tensor([[1, 2, 3],
        [4, 5, 6]])
x shape: torch.Size([2, 3])
x data_ptr: 312203904
is contiguous: True
view_x : tensor([[1, 2],
        [3, 4],
        [5, 6]])
view_x shape : torch.Size([3, 2])
view_x data_ptr: 312203904
is contiguous: True
y: tensor([[1, 4],
        [2, 5],
        [3, 6]])
y shape: torch.Size([3, 2])
y data_ptr: 312203904
is contiguous: False
view_y : tensor([[1, 4, 2],
        [5, 3, 6]])
view_y shape : torch.Size([2, 3])
view_y data_ptr: 308175168
is contiguous: True


In [ ]:
#einsum
x = torch.ones(2,3,4)
y = torch.zeros(2,3,4)

print(x@y.transpose(-2,-1))

print("="*20)
print("="*20)

z = torch.einsum("bih, bjh -> bij", x, y)

print(z)

tensor([[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]])
tensor([[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]])


In [ ]:
x = torch.ones((3,2))

print(torch.sum(x, dim=-1))

y = torch.einsum("ij->i", x)

print(y)

tensor([2., 2., 2.])
tensor([2., 2., 2.])


In [ ]:
x = torch.rand((2,3,4))

print(x.view(2,3,2,2))

y = torch.einsum("bi(hd)->bihd",x,h=2)
print(y)

tensor([[[[0.3545, 0.3413],
          [0.0881, 0.1504]],

         [[0.9531, 0.5945],
          [0.6175, 0.5552]],

         [[0.4119, 0.3801],
          [0.6527, 0.8257]]],


        [[[0.1728, 0.4558],
          [0.2169, 0.8597]],

         [[0.2089, 0.3893],
          [0.7296, 0.1873]],

         [[0.5850, 0.9695],
          [0.8662, 0.0961]]]])


TypeError: einsum() got an unexpected keyword argument 'h'

In [ ]:
from einops import reduce, rearrange

# sum
x = torch.ones((3,2))

y = reduce(x, "seq hidden -> seq", "sum")
print(y)

# rearrange view
x = torch.rand((2,3,4))
y = rearrange(x, "batch seq (heads dim) -> batch seq heads dim", heads=2)
print(y)
print(y.shape)

tensor([2., 2., 2.])
tensor([[[[0.7121, 0.5938],
          [0.6151, 0.4469]],

         [[0.0111, 0.2133],
          [0.7358, 0.4031]],

         [[0.6786, 0.0082],
          [0.9380, 0.0954]]],


        [[[0.6680, 0.6506],
          [0.8708, 0.7235]],

         [[0.5637, 0.0639],
          [0.7238, 0.4178]],

         [[0.6021, 0.1409],
          [0.1720, 0.1616]]]])
torch.Size([2, 3, 2, 2])


In [ ]:
# weight initialization
torch.manual_seed(123)
input_dim = 1024*16
hidden_dim = 32

w = nn.Parameter(torch.randn(input_dim, hidden_dim))
x = nn.Parameter(torch.randn(input_dim))

output = x @ w

output #output scales with increase in size of input_dim

tensor([ 266.8163,   26.5456,   58.2681,  138.2336,   15.7877,  -75.0963,
         157.9305,  -67.7599,  -12.2784, -174.5006, -123.4288,   94.0587,
        -126.0809,  261.5685,  -99.6406, -222.2314,  169.3665,   84.8937,
         -11.3559,   96.3601, -155.9946,  -86.4659,  114.0224,  -21.6987,
          51.3609, -101.3496,  103.3351,  -57.1707,   52.5049,  -76.7859,
        -253.5217,   45.4793], grad_fn=<SqueezeBackward4>)

In [ ]:
# solution divide by sqrt(input_dim)
torch.manual_seed(123)
input_dim = 1024*16
hidden_dim = 32

w = nn.Parameter(torch.randn(input_dim, hidden_dim)/(input_dim)**0.5)
x = nn.Parameter(torch.randn(input_dim))

output = x @ w

output

tensor([ 2.0845,  0.2074,  0.4552,  1.0800,  0.1233, -0.5867,  1.2338, -0.5294,
        -0.0959, -1.3633, -0.9643,  0.7348, -0.9850,  2.0435, -0.7784, -1.7362,
         1.3232,  0.6632, -0.0887,  0.7528, -1.2187, -0.6755,  0.8908, -0.1695,
         0.4013, -0.7918,  0.8073, -0.4466,  0.4102, -0.5999, -1.9806,  0.3553],
       grad_fn=<SqueezeBackward4>)

In [ ]:
#optimizer AdaGrad

class AdaGrad(torch.optim.Optimizer):
  def __init__(self, params, lr=0.01):
    super(AdaGrad, self).__init__(params, dict(lr=lr))

  def step(self):
    for group in self.param_groups:
      lr = group["lr"]
      for p in group["params"]:
        #optimizer state
        state = self.state[p]
        grad = p.grad.data

        #get squared gradients g2
        g2 = state.get("g2", torch.zeros_like(grad))

        #update optimizer state
        g2 += torch.square(grad)
        state["g2"] = g2

        #update parameters
        p.data -= lr * grad/torch.sqrt(g2 + 1e-5)

In [ ]:
x = torch.tensor([1,4,9])

torch.sqrt(x)

tensor([1., 2., 3.])

# Pre-norm vs Post-norm

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
device

'cpu'

In [4]:
#lets implement compact class for normalization
class LayerNorm(nn.Module):
  def __init__(self, emb_size):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_size))
    self.shift = nn.Parameter(torch.zeros(emb_size))

  def forward(self, x):
    mean = x.mean(keepdim=True, dim=-1)
    variance = x.var(keepdim=True, dim=-1, unbiased=False)
    norm_x = (x - mean)/torch.sqrt(variance + self.eps)
    return self.scale * norm_x + self.shift

class MultiHeadAttention(nn.Module):
  def __init__(self, dim_in, dim_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (dim_out % num_heads == 0), "dim_out must be divisible by num_heads"

    self.dim_out = dim_out # final merged context vector embedding size
    self.num_heads = num_heads
    self.head_dim = dim_out//num_heads # embedding size of context vector in single head
    self.w_query = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
    self.w_key = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
    self.w_value = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
    self.out_proj = torch.nn.Linear(dim_out, dim_out) # transform merged context_vectors into similar dimension size vectors
    self.dropout = torch.nn.Dropout(dropout)
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    batch_size, num_tokens, dim_in = x.shape
    queries = self.w_query(x)
    keys = self.w_key(x)
    values = self.w_value(x)  #shape (batch_size, num_tokens, dim_out)

    queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim) #shape (batch_size, num_tokens, num_heads, head_dim)

    queries = queries.transpose(1,2)
    keys = keys.transpose(1,2)
    values = values.transpose(1,2) #shape (batch_size, num_heads, num_tokens, head_dim)

    attention_scores = queries @ keys.transpose(2,3)
    attention_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens], -torch.inf)

    attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
    attention_weights = self.dropout(attention_weights)

    context_vectors = (attention_weights @ values).transpose(1,2) #transposing axis 1,2  since we have to merge the context vectors by num_heads and head_dim, so required shape will now be (batch_size, num_tokens, num_heads, head_dim)
    context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.dim_out)

    context_vectors = self.out_proj(context_vectors)

    return context_vectors

class GeLU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x,3))
    ))

class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg['emb_size'], 4 * cfg['emb_size']),
        GeLU(),
        nn.Linear(4 * cfg['emb_size'], cfg['emb_size'])
    )

  def forward(self, x):
    return self.layers(x)



In [5]:
GPT_CONFIG_124M = {
    'emb_size':768,
    'context_length':1024,
    'vocab_size':50257,
    'num_heads':12,
    'num_layers':12,
    'drop_rate':0.1,
    'qkv_bias':False,
}

In [9]:
torch.manual_seed(123)
tokens = torch.tensor([23, 44, 513, 34, 2343, 467, 842, 643], device=device).unsqueeze(0)
target = torch.tensor([44, 513, 34, 2343, 467, 842, 643, 4323], device=device).long().unsqueeze(0)
embedding_layer = nn.Embedding(GPT_CONFIG_124M["vocab_size"], GPT_CONFIG_124M["emb_size"]).to(device)

inp_embedding = embedding_layer(tokens).detach()

In [12]:
#pre-norm transformer
class PrenormTransformer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.layer_norm1 = LayerNorm(config['emb_size'])
    self.layer_norm2 = LayerNorm(config['emb_size'])
    self.mha = MultiHeadAttention(config['emb_size'], config['emb_size'], config['context_length'], config['drop_rate'], config['num_heads'], qkv_bias=config['qkv_bias'])
    self.ffn = FeedForward(config)
    self.dropout = nn.Dropout(config['drop_rate'])
    self.proj = nn.Linear(config['emb_size'], config['vocab_size'], bias=False)

  def forward(self, x):
    res = x
    x = self.layer_norm1(x)
    x = self.mha(x)
    x = self.dropout(x)
    x = x + res

    res = x
    x = self.layer_norm2(x)
    x = self.ffn(x)
    x = self.dropout(x)
    x = x + res
    x= self.proj(x)

    return x

In [13]:
torch.manual_seed(123)
prenorm = PrenormTransformer(GPT_CONFIG_124M).to(device)

logits1 = prenorm(inp_embedding)
print(logits1.flatten(0, 1).shape, target.flatten(0,1).shape)
loss1 = nn.functional.cross_entropy(logits1.flatten(0, 1), target.flatten(0, 1))
loss1.backward()


torch.Size([8, 50257]) torch.Size([8])


In [14]:
for name, p in prenorm.named_parameters():
  if p.requires_grad:
    print(f"Layer {name} grads: {torch.mean(torch.abs(p.grad))}")

Layer layer_norm1.scale grads: 0.0013279084814712405
Layer layer_norm1.shift grads: 0.0021793029736727476
Layer layer_norm2.scale grads: 0.0012161083286628127
Layer layer_norm2.shift grads: 0.0013114080065861344
Layer mha.w_query.weight grads: 0.00044653398799709976
Layer mha.w_key.weight grads: 0.00046367497998289764
Layer mha.w_value.weight grads: 0.0021524035837501287
Layer mha.out_proj.weight grads: 0.0021978961303830147
Layer mha.out_proj.bias grads: 0.006587665993720293
Layer ffn.layers.0.weight grads: 0.0010093546006828547
Layer ffn.layers.0.bias grads: 0.0010972961317747831
Layer ffn.layers.2.weight grads: 0.001970293466001749
Layer ffn.layers.2.bias grads: 0.00643738592043519
Layer proj.weight grads: 2.3798231268301606e-05


In [15]:
#post-norm transformer
class PostnormTransformer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.layer_norm1 = LayerNorm(config['emb_size'])
    self.layer_norm2 = LayerNorm(config['emb_size'])
    self.mha = MultiHeadAttention(config['emb_size'], config['emb_size'], config['context_length'], config['drop_rate'], config['num_heads'], qkv_bias=config['qkv_bias'])
    self.ffn = FeedForward(config)
    self.dropout = nn.Dropout(config['drop_rate'])
    self.proj = nn.Linear(config['emb_size'], config['vocab_size'], bias=False)

  def forward(self, x):
    res = x
    x = self.mha(x)
    x = self.dropout(x)
    x = x + res
    x = self.layer_norm1(x)

    res = x
    x = self.ffn(x)
    x = self.dropout(x)
    x = x + res
    x = self.layer_norm2(x)
    x= self.proj(x)

    return x

In [19]:
torch.manual_seed(123)
postnorm = PostnormTransformer(GPT_CONFIG_124M).to(device)

logits2 = postnorm(inp_embedding)
loss2 = nn.functional.cross_entropy(logits2.flatten(0, 1), target.flatten(0, 1))
loss2.backward()


In [20]:
for name, p in postnorm.named_parameters():
  if p.requires_grad:
    print(f"Layer {name} grads: {torch.mean(torch.abs(p.grad))}")

Layer layer_norm1.scale grads: 0.006016042549163103
Layer layer_norm1.shift grads: 0.006018439307808876
Layer layer_norm2.scale grads: 0.005978296976536512
Layer layer_norm2.shift grads: 0.006049750838428736
Layer mha.w_query.weight grads: 0.0004198396054562181
Layer mha.w_key.weight grads: 0.00043614854803308845
Layer mha.w_value.weight grads: 0.00202961009927094
Layer mha.out_proj.weight grads: 0.0020719249732792377
Layer mha.out_proj.bias grads: 0.006333619821816683
Layer ffn.layers.0.weight grads: 0.0009835539385676384
Layer ffn.layers.0.bias grads: 0.0010712787043303251
Layer ffn.layers.2.weight grads: 0.0019210135797038674
Layer ffn.layers.2.bias grads: 0.006283838301897049
Layer proj.weight grads: 2.273203972436022e-05


the difference is grad can be seen when number of layers increase